[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# The Aggregation Pipeline &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's boot cell, which also builds the three baskets. Run it first. Each
task opens its own client and closes it, so they can be run in any order.


In [1]:
import os
import random
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import pymongo

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")

def failed(error):
    """A failure's real message, without the cluster time that changes every run."""
    details = getattr(error, "details", None) or {}
    message = details.get("errmsg", str(error).split(", full error")[0])
    return f"{type(error).__name__}: {message.split(' :: caused by :: ')[-1]}"


def work_done(pipeline, collection="products"):
    """What the server had to read to run a pipeline, which is what stage order decides."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        shop = client.get_default_database()
        explained = shop.command("explain",
                                 {"aggregate": collection, "pipeline": pipeline, "cursor": {}},
                                 verbosity="executionStats")
        stats = explained.get("executionStats")
        if stats is None:                                           # a pipeline with stages after it
            stats = explained["stages"][0]["$cursor"]["executionStats"]
        return {"documents": stats["totalDocsExamined"], "index keys": stats["totalKeysExamined"]}


def build_baskets():
    """Three baskets: one with items, one with an empty array, one with no such field."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        shop = client.get_default_database()
        shop.baskets.drop()
        shop.baskets.insert_many([{"_id": 1, "owner": "ana", "items": ["a", "b"]},
                                  {"_id": 2, "owner": "bo", "items": []},
                                  {"_id": 3, "owner": "cy"}])
        return shop.baskets.count_documents({})


print("server: ", start_server())
print("replica:", initiate())
print("seeded: ", seed(), "products")
print("baskets:", build_baskets())
print(report())


server:  already running
replica: replica set rs0, primary
seeded:  500 products
baskets: 3
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 500


**1.** How many of each maker.


In [2]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

for row in shop.products.aggregate([
        {"$group": {"_id": "$maker", "n": {"$sum": 1}}},
        {"$sort": {"_id": 1}}]):
    print(f"  {row['_id']:8} {row['n']}")
client.close()


  Aster    125
  Belden   125
  Corvid   125
  Dalgo    125


`$sum: 1` counts, because it adds one for every document in the group. The `$sort` on `_id` is what
makes the output the same order every time.


**2.** One row for everything.


In [3]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

print(list(shop.products.aggregate([
    {"$group": {"_id": None, "stock": {"$sum": "$stock"}, "average": {"$avg": "$price"}}},
    {"$project": {"_id": 0, "stock": 1, "average": {"$round": ["$average", 2]}}},
])))
client.close()


[{'stock': 99019, 'average': 1005.54}]


`{"_id": None}` is how you say "do not group by anything". The `$round` is in a `$project` because
accumulators produce the value and `$project` is where it gets presented.


**3.** The cheapest of each kind.


In [4]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

for row in shop.products.aggregate([
        {"$sort": {"price": 1}},
        {"$group": {"_id": "$kind", "name": {"$first": "$name"},
                    "price": {"$first": "$price"}}},
        {"$sort": {"_id": 1}}]):
    print(f"  {row['_id']:9} {row['price']:8.2f}  {row['name']}")
client.close()


  cable         7.47  Dalgo cable 399
  keyboard     41.47  Belden keyboard 277
  laptop       58.26  Dalgo laptop 15
  monitor       7.34  Dalgo monitor 391
  mouse         9.43  Belden mouse 413


The first `$sort` decides what `$first` means, and the second sorts the five groups for display.
Two sorts doing two different jobs, and swapping them would break the answer rather than the order.


**4.** Every tag, counted.


In [5]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

for row in shop.products.aggregate([
        {"$unwind": "$tags"},
        {"$group": {"_id": "$tags", "n": {"$sum": 1}}},
        {"$sort": {"_id": 1}}]):
    print(f"  {row['_id']:12} {row['n']}")
client.close()


  bulk         196
  clearance    178
  new          212
  refurbished  194
  sale         220


Every product has exactly two tags, so the counts add up to twice the collection. `$unwind` is the
only way to count array elements rather than the documents holding them.


**5.** What $unwind keeps.


In [6]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
build_baskets()

plain = list(shop.baskets.aggregate([{"$unwind": "$items"}]))
kept = list(shop.baskets.aggregate([
    {"$unwind": {"path": "$items", "preserveNullAndEmptyArrays": True}}]))

print("baskets:            ", shop.baskets.count_documents({}))
print("plain $unwind rows: ", len(plain), "from", sorted({row["_id"] for row in plain}))
print("preserving rows:    ", len(kept), "from", sorted({row["_id"] for row in kept}))
client.close()


baskets:             3
plain $unwind rows:  2 from [1]
preserving rows:     4 from [1, 2, 3]


The empty basket and the one with no `items` field at all are both gone from the plain form, and
both present in the preserving one with no `items` key.


**6.** Where the $match goes.


In [7]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
shop.products.create_index([("kind", 1), ("price", 1)], name="kind_price")

print("$match first:", work_done([
    {"$match": {"kind": "mouse"}},
    {"$group": {"_id": "$maker", "n": {"$sum": 1}}}]))
print("$match last: ", work_done([
    {"$group": {"_id": "$maker", "kinds": {"$addToSet": "$kind"}}},
    {"$match": {"kinds": "mouse"}}]))
client.close()


$match first: {'documents': 100, 'index keys': 100}
$match last:  {'documents': 500, 'index keys': 0}


A hundred documents read against five hundred, and a hundred index keys against none. After a
`$group` the documents are new ones the server built, and no index describes them.


---

&#8592; **Back to:** [The Aggregation Pipeline](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/09-the-aggregation-pipeline.ipynb)  &nbsp;&middot;&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)
